# 02 — Preparar perfiles de voz XTTS

Notebook local para crear perfiles reutilizables a partir de grabaciones autorizadas. **No entrena un modelo nuevo**: XTTS calcula representaciones de voz y las guarda para usarlas al sintetizar textos nuevos.

Cada persona tiene un identificador propio. Ejemplo: el audio `data/voice_samples/winz.wav` produce el perfil `data/voice_profiles/winz.pt`. Para otra persona se cambia `VOICE_ID` y se repite el proceso.

## 1. Cargar XTTS y comprobar CUDA

In [1]:
import os
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)
print('CUDA disponible:', torch.cuda.is_available())


ModuleNotFoundError: No module named 'torch'

## 2. Cargar XTTS-v2
PyTorch reciente restringe los objetos que puede reconstruir desde checkpoints. Aquí permitimos únicamente la clase de configuración de XTTS que requiere el checkpoint oficial.


In [ ]:
import torch
from TTS.tts.configs.xtts_config import XttsConfig
torch.serialization.add_safe_globals([XttsConfig])

from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(DEVICE)
model = tts.synthesizer.tts_model
print('XTTS cargado')


## 3. Elegir el perfil y su grabación

Usa una grabación limpia de la persona cuyo perfil vas a crear. Cada persona tiene su propio archivo WAV en `data/voice_samples/`, identificado por nombre o alias.

Graba entre 10 y 30 segundos en un lugar silencioso, con voz natural y sin música. Puedes reutilizar el mismo guion para todas las personas; graba cada perfil por separado y solo con autorización de esa persona.

In [ ]:
# Cambia este identificador para crear otro perfil (ej.: "winz", "persona_02").
VOICE_ID = "winz"

from pathlib import Path
VOICE_SAMPLES_DIR = Path("data/voice_samples")
VOICE_PROFILES_DIR = Path("data/voice_profiles")
voice_sample_path = VOICE_SAMPLES_DIR / f"{VOICE_ID}.wav"
assert voice_sample_path.is_file(), f"No existe la grabación: {voice_sample_path}"
print("Persona seleccionada:", VOICE_ID)
print("Audio seleccionado:", voice_sample_path)

## 4. Crear el perfil de voz

XTTS calcula dos representaciones internas a partir de la grabación de esta persona. Guardarlas permite reutilizar el perfil sin recalcularlas en cada sesión; XTTS sigue siendo necesario para sintetizar.

In [ ]:
with torch.inference_mode():
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=[str(voice_sample_path)],
        gpt_cond_len=6,
        max_ref_length=30,
        sound_norm_refs=False
    )
print("Perfil calculado:", VOICE_ID)
print("Dimensiones:", tuple(gpt_cond_latent.shape), tuple(speaker_embedding.shape))

## 5. Guardar el perfil

El perfil se guarda con el mismo identificador que el audio. Mantén los audios y perfiles fuera del repositorio público.

In [ ]:
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
profile_path = VOICE_PROFILES_DIR / f"{VOICE_ID}.pt"
torch.save({
    "voice_id": VOICE_ID,
    "gpt_cond_latent": gpt_cond_latent.detach().cpu(),
    "speaker_embedding": speaker_embedding.detach().cpu(),
    "model": "XTTS-v2",
    "language": "es"
}, profile_path)
print("Perfil guardado:", profile_path)

## 6. Probar una frase nueva

La frase generada debe ser distinta de la grabación de referencia. La prueba carga el perfil recién guardado y usa la inferencia de XTTS.

In [ ]:
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = "Hola, esta es una prueba de síntesis con mi perfil de voz."
with torch.inference_mode():
    result = model.inference(
        text=text, language="es",
        gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
        speaker_embedding=profile["speaker_embedding"].to(DEVICE),
        enable_text_splitting=True
    )
from IPython.display import Audio, display
display(Audio(result["wav"], rate=24000))

## 7. Reutilizar perfiles en el asistente de compras

Copia los archivos `.pt` a la carpeta de perfiles de Google Drive configurada en `Lab4_Asistente_Compras_por_Voz_MVP.ipynb`. En ese notebook, cambia `VOICE_ID` para seleccionar qué persona usará XTTS. Por ejemplo, `winz.pt` y `persona_02.pt`.

**Privacidad:** la grabación y el perfil pueden reproducir una voz identificable. Guárdalos en un espacio privado y usa solo voces propias o autorizadas.